# 03 — Pipeline ML Reproductible + GPU Setup
**Projet LendingClub | Membre 1**

| Étape | Contenu |
|---|---|
| 1 | Vérification GPU NVIDIA |
| 2 | Installation RAPIDS cuML (GPU) |
| 3 | Pipeline sklearn reproductible |
| 4 | ColumnTransformer (num + cat) |
| 5 | Sauvegarde pipeline + données train/test |

---
## 🎮 Utiliser ta carte NVIDIA

Deux librairies permettent d'utiliser le GPU pour le ML :

| Librairie | Utilité | Speedup |
|---|---|---|
| **RAPIDS cuML** | KNN, SVM, PCA, UMAP, K-Means sur GPU | 10–100× |
| **LightGBM GPU** | Gradient boosting sur GPU | 3–10× |
| **XGBoost GPU** | Gradient boosting sur GPU | 3–10× |

Le notebook détecte automatiquement le GPU et bascule dessus si disponible.

## 1. Vérification du GPU NVIDIA

In [1]:
import subprocess, sys

print('=== VÉRIFICATION GPU NVIDIA ===')
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
except FileNotFoundError:
    print('⚠️  nvidia-smi non trouvé — GPU non détecté')
    GPU_AVAILABLE = False

print(f'\nGPU disponible : {GPU_AVAILABLE}')

=== VÉRIFICATION GPU NVIDIA ===
Mon Apr  6 00:37:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P8              1W /  140W |     247MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------

In [2]:
# Vérification via PyTorch (souvent déjà installé)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        print(f'✅ PyTorch CUDA disponible')
        print(f'   GPU : {torch.cuda.get_device_name(0)}')
        print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    else:
        print('PyTorch installé mais CUDA non disponible')
except ImportError:
    print('PyTorch non installé — on utilisera cuML directement')

PyTorch installé mais CUDA non disponible


## 2. Installation & Configuration GPU pour ML

### Option A — RAPIDS cuML (recommandé pour ce projet)
RAPIDS est la librairie NVIDIA pour le ML sur GPU.
Elle réimplémente les algorithmes sklearn mais sur GPU.

```bash
# À exécuter UNE SEULE FOIS dans un terminal Anaconda :
# (adapter cuda12 selon ta version de CUDA)
pip install --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12
```

### Option B — LightGBM + XGBoost GPU (plus simple)
Ces librairies supportent le GPU nativement via un paramètre.

In [3]:
# ── Détection RAPIDS cuML ─────────────────────────────────────────────
CUML_AVAILABLE = False
try:
    import cuml
    CUML_AVAILABLE = True
    print(f'✅ RAPIDS cuML disponible — version {cuml.__version__}')
    print('   → KNN, SVM, UMAP, K-Means seront exécutés sur GPU')
except ImportError:
    print('ℹ️  RAPIDS cuML non installé — sklearn CPU sera utilisé')
    print('   Pour installer RAPIDS : voir instructions dans la cellule précédente')

# ── Détection LightGBM GPU ────────────────────────────────────────────
LGBM_GPU = False
try:
    import lightgbm as lgb
    # Test si le GPU device est disponible
    if GPU_AVAILABLE:
        LGBM_GPU = True
        print(f'✅ LightGBM {lgb.__version__} — GPU activé via device="gpu"')
    else:
        print(f'ℹ️  LightGBM {lgb.__version__} — CPU uniquement')
except ImportError:
    print('ℹ️  LightGBM non installé — pip install lightgbm')

# ── Configuration globale ─────────────────────────────────────────────
USE_GPU = GPU_AVAILABLE
print(f'\n🎮 Configuration finale : USE_GPU = {USE_GPU}')

ℹ️  RAPIDS cuML non installé — sklearn CPU sera utilisé
   Pour installer RAPIDS : voir instructions dans la cellule précédente
✅ LightGBM 4.6.0 — GPU activé via device="gpu"

🎮 Configuration finale : USE_GPU = True


In [4]:
# ── Paramètres GPU pour LightGBM ─────────────────────────────────────
# Ces paramètres seront utilisés par le Membre 2 pour ses modèles

LGBM_GPU_PARAMS = {
    'device'          : 'gpu' if USE_GPU else 'cpu',
    'gpu_platform_id' : 0,
    'gpu_device_id'   : 0,
    'n_jobs'          : -1,
}

XGBOOST_GPU_PARAMS = {
    'device'    : 'cuda' if USE_GPU else 'cpu',  # XGBoost >= 2.0
    'tree_method': 'hist',                        # requis pour GPU
    'n_jobs'    : -1,
}

print('Paramètres GPU LightGBM :', LGBM_GPU_PARAMS)
print('Paramètres GPU XGBoost  :', XGBOOST_GPU_PARAMS)
print('\n→ Ces paramètres seront importés dans 04_classification.ipynb')

Paramètres GPU LightGBM : {'device': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0, 'n_jobs': -1}
Paramètres GPU XGBoost  : {'device': 'cuda', 'tree_method': 'hist', 'n_jobs': -1}

→ Ces paramètres seront importés dans 04_classification.ipynb


## 3. Chargement & Séparation train/test

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os, json
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed'

df = pd.read_parquet(f'{PROCESSED}/lending_features.parquet')
print(f'✅ Dataset chargé : {df.shape[0]:,} × {df.shape[1]}')
print(f'Taux de défaut : {df["target"].mean()*100:.1f}%')

✅ Dataset chargé : 1,345,310 × 128
Taux de défaut : 20.0%


In [6]:
from sklearn.model_selection import train_test_split

# Séparer features et target
# Exclure colonnes catégorielles brutes (gardées pour CatBoost uniquement)
CAT_BRUTES = ['purpose','addr_state','home_ownership','verification_status',
              'application_type','initial_list_status','title','zip_code','emp_title']

# On garde les colonnes catégorielles encodées (_te) et on exclut les brutes
# sauf pour CatBoost qui les gère nativement
cat_brutes_present = [c for c in CAT_BRUTES if c in df.columns]

X_full = df.drop(columns=['target'] + cat_brutes_present)
X_cat  = df.drop(columns=['target'])   # version avec catégorielles pour CatBoost
y      = df['target']

print(f'X_full (encodé) : {X_full.shape}')
print(f'X_cat  (brut)   : {X_cat.shape}')
print(f'y               : {y.shape}')

X_full (encodé) : (1345310, 118)
X_cat  (brut)   : (1345310, 127)
y               : (1345310,)


In [7]:
# Train / Test split stratifié (80% / 20%)
# stratify=y → préserve le ratio 80/20 dans les deux splits
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train  : {X_train.shape[0]:>10,} lignes  ({y_train.mean()*100:.1f}% défaut)')
print(f'Test   : {X_test.shape[0]:>10,} lignes  ({y_test.mean()*100:.1f}% défaut)')
print('✅ Ratio de défaut préservé dans les deux splits (stratify)')

Train  :  1,076,248 lignes  (20.0% défaut)
Test   :    269,062 lignes  (20.0% défaut)
✅ Ratio de défaut préservé dans les deux splits (stratify)


## 4. Pipeline sklearn reproductible ★

**Pourquoi un Pipeline ?**
- Garantit que le StandardScaler est fitté UNIQUEMENT sur les données d'entraînement
  (évite un data leakage subtil)
- Reproductible : une seule ligne pour entraîner, une pour prédire
- Exportable : `joblib.dump(pipeline, 'model.pkl')` → déploiement Streamlit

In [8]:
from sklearn.pipeline         import Pipeline
from sklearn.compose          import ColumnTransformer
from sklearn.preprocessing    import StandardScaler, OrdinalEncoder
from sklearn.impute            import SimpleImputer
from sklearn.experimental      import enable_iterative_imputer
from sklearn.impute            import IterativeImputer

# ── Identifier les types de colonnes ─────────────────────────────────
num_features = X_train.select_dtypes(include=np.number).columns.tolist()
cat_features = X_train.select_dtypes(include='object').columns.tolist()

print(f'Features numériques : {len(num_features)}')
print(f'Features catégorielles : {len(cat_features)}')
if cat_features:
    print(f'  Catégorielles : {cat_features}')

Features numériques : 114
Features catégorielles : 4
  Catégorielles : ['grade', 'sub_grade', 'pymnt_plan', 'disbursement_method']


In [9]:
# ── Sous-pipeline numérique ───────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # sécurité si NaN restants
    ('scaler',  StandardScaler()),                  # centrage-réduction
])

# ── Sous-pipeline catégoriel ──────────────────────────────────────────
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

# ── ColumnTransformer : applique chaque pipeline aux bonnes colonnes ──
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer,     num_features),
        ('cat', categorical_transformer, cat_features),
    ],
    remainder='drop',   # supprimer les colonnes non spécifiées
    verbose_feature_names_out=False
)

print('✅ ColumnTransformer configuré')
print(f'   Numériques  : {len(num_features)} colonnes → StandardScaler')
print(f'   Catégorielles: {len(cat_features)} colonnes → OrdinalEncoder')

✅ ColumnTransformer configuré
   Numériques  : 114 colonnes → StandardScaler
   Catégorielles: 4 colonnes → OrdinalEncoder


In [10]:
# ── Test du preprocessor ─────────────────────────────────────────────
import time
print('⏳ Fit du preprocessor sur X_train ...')
t0 = time.time()
preprocessor.fit(X_train)
print(f'✅ Fit terminé en {time.time()-t0:.1f}s')

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed  = preprocessor.transform(X_test)

print(f'\nX_train après pipeline : {X_train_transformed.shape}')
print(f'X_test  après pipeline : {X_test_transformed.shape}')
print(f'NaN dans X_train       : {np.isnan(X_train_transformed).sum()}')
print(f'NaN dans X_test        : {np.isnan(X_test_transformed).sum()}')

⏳ Fit du preprocessor sur X_train ...
✅ Fit terminé en 20.2s

X_train après pipeline : (1076248, 118)
X_test  après pipeline : (269062, 118)
NaN dans X_train       : 0
NaN dans X_test        : 0


## 5. Sauvegarde

In [11]:
import joblib

# ── Sauvegarder preprocessor ─────────────────────────────────────────
joblib.dump(preprocessor, f'{PROCESSED}/preprocessor.pkl')
print(f'✅ preprocessor.pkl sauvegardé')

# ── Sauvegarder splits ───────────────────────────────────────────────
X_train.to_parquet(f'{PROCESSED}/X_train.parquet', index=False)
X_test.to_parquet (f'{PROCESSED}/X_test.parquet',  index=False)
y_train.to_frame().to_parquet(f'{PROCESSED}/y_train.parquet', index=False)
y_test.to_frame().to_parquet (f'{PROCESSED}/y_test.parquet',  index=False)
print(f'✅ X_train / X_test / y_train / y_test sauvegardés')

# ── Sauvegarder la liste des features ────────────────────────────────
feature_info = {
    'num_features'  : num_features,
    'cat_features'  : cat_features,
    'all_features'  : num_features + cat_features,
    'n_features'    : len(num_features) + len(cat_features),
    'use_gpu'       : USE_GPU,
    'lgbm_gpu_params'   : LGBM_GPU_PARAMS,
    'xgboost_gpu_params': XGBOOST_GPU_PARAMS,
}
import json
with open(f'{PROCESSED}/feature_info.json', 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f'✅ feature_info.json sauvegardé')

✅ preprocessor.pkl sauvegardé
✅ X_train / X_test / y_train / y_test sauvegardés
✅ feature_info.json sauvegardé


In [12]:
# ── Résumé final ─────────────────────────────────────────────────────
print('=' * 55)
print('   RÉSUMÉ — 03_PIPELINE')
print('=' * 55)
print(f'  GPU disponible         : {USE_GPU}')
print(f'  RAPIDS cuML            : {CUML_AVAILABLE}')
print(f'  LightGBM GPU           : {LGBM_GPU}')
print(f'  Features totales       : {len(num_features)+len(cat_features)}')
print(f'  Features numériques    : {len(num_features)}')
print(f'  Features catégorielles : {len(cat_features)}')
print(f'  X_train                : {X_train.shape}')
print(f'  X_test                 : {X_test.shape}')
print(f'  Taux défaut train      : {y_train.mean()*100:.1f}%')
print(f'  Taux défaut test       : {y_test.mean()*100:.1f}%')
print('=' * 55)
print('\nFichiers produits :')
for f in ['preprocessor.pkl','X_train.parquet','X_test.parquet',
          'y_train.parquet','y_test.parquet','feature_info.json']:
    path = f'{PROCESSED}/{f}'
    if os.path.exists(path):
        print(f'  ✅ {path}  ({os.path.getsize(path)/1e6:.1f} MB)')

print('\n→ Le Membre 2 peut maintenant démarrer : 04_classification.ipynb')
print('→ Paramètres GPU exportés dans feature_info.json')

   RÉSUMÉ — 03_PIPELINE
  GPU disponible         : True
  RAPIDS cuML            : False
  LightGBM GPU           : True
  Features totales       : 118
  Features numériques    : 114
  Features catégorielles : 4
  X_train                : (1076248, 118)
  X_test                 : (269062, 118)
  Taux défaut train      : 20.0%
  Taux défaut test       : 20.0%

Fichiers produits :
  ✅ ../data/processed/preprocessor.pkl  (0.0 MB)
  ✅ ../data/processed/X_train.parquet  (151.7 MB)
  ✅ ../data/processed/X_test.parquet  (38.4 MB)
  ✅ ../data/processed/y_train.parquet  (0.2 MB)
  ✅ ../data/processed/y_test.parquet  (0.0 MB)
  ✅ ../data/processed/feature_info.json  (0.0 MB)

→ Le Membre 2 peut maintenant démarrer : 04_classification.ipynb
→ Paramètres GPU exportés dans feature_info.json


---
## Annexe — Guide complet GPU pour le Membre 2

### Comment utiliser le GPU dans LightGBM
```python
import lightgbm as lgb
import json

# Charger les paramètres GPU générés par le Membre 1
with open('../data/processed/feature_info.json') as f:
    info = json.load(f)

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    **info['lgbm_gpu_params']   # device='gpu' si GPU disponible
)
```

### Comment utiliser le GPU dans XGBoost
```python
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    **info['xgboost_gpu_params']   # device='cuda' si GPU disponible
)
```

### Comment utiliser RAPIDS cuML (si installé)
```python
# Remplacement drop-in de sklearn — même API !
if CUML_AVAILABLE:
    from cuml.cluster import KMeans          # K-Means sur GPU
    from cuml.manifold import UMAP           # UMAP sur GPU
    from cuml.svm import SVC                 # SVM sur GPU
else:
    from sklearn.cluster import KMeans
    from umap import UMAP
    from sklearn.svm import SVC
```

### Vérifier la vitesse GPU vs CPU
```python
import time
# CPU
t0 = time.time(); model_cpu.fit(X_train, y_train); print(f'CPU: {time.time()-t0:.0f}s')
# GPU  
t0 = time.time(); model_gpu.fit(X_train, y_train); print(f'GPU: {time.time()-t0:.0f}s')
```